In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import timm
import numpy as np
from collections import Counter
from PIL import Image
from scipy.io import loadmat

from torchvision import models

In [43]:
# ========== 自监督学习 DINO 部分 ==========
class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        # 使用 DINO 的 ResNet50 预训练模型
        self.encoder = timm.create_model('resnet50', pretrained=True)
        self.encoder.fc = nn.Identity()  # 去掉最后的分类层

    def forward(self, x):
        return self.encoder(x)

class TeacherModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('resnet50', pretrained=True, num_classes=0)
        self.encoder.fc = nn.Identity()
    
    def forward(self, x):
        return self.encoder(x)

class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = models.resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()
    
    def forward(self, x):
        return self.encoder(x)

class MultiModalDistillationModel(nn.Module):
    def __init__(self, teacher_encoder, student_encoder, nir_encoder, fusion, num_classes):
        super().__init__()
        self.teacher_encoder = teacher_encoder
        self.student_encoder = student_encoder
        self.nir_encoder = nir_encoder
        self.fusion = fusion
        self.classifier = nn.Sequential(
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, num_classes))

    def forward(self, img, nir):
        # 学生网络
        student_img_feat = self.student_encoder(img)
        # 教师网络
        with torch.no_grad():
            teacher_img_feat = self.teacher_encoder(img)  # 只传img
        
        # NIR特征
        nir_feat = self.nir_encoder(nir)
        
        # 融合学生和NIR的特征
        fused_feat = self.fusion(student_img_feat, nir_feat)
        
        logits = self.classifier(fused_feat)
        return logits, student_img_feat, teacher_img_feat

def update_teacher_model(student_model, teacher_model, beta=0.99):
    # 滑动平均更新教师模型参数
    with torch.no_grad():
        for student_params, teacher_params in zip(student_model.parameters(), teacher_model.parameters()):
            teacher_params.data = beta * teacher_params.data + (1.0 - beta) * student_params.data

def contrastive_loss(student_features, teacher_features, temperature=0.07):
    # 计算学生和教师特征的相似度
    student_features = nn.functional.normalize(student_features, p=2, dim=-1)
    teacher_features = nn.functional.normalize(teacher_features, p=2, dim=-1)
    
    # 计算学生特征和教师特征之间的对比损失（这里使用的是简化版的对比损失）
    logits = torch.matmul(student_features, teacher_features.T) / temperature
    labels = torch.arange(student_features.size(0)).to(student_features.device)
    loss = nn.functional.cross_entropy(logits, labels)
    return loss

In [44]:
# ========== 蒸馏损失 ==========
def distillation_loss(student_outputs, teacher_outputs, temperature=2.0, alpha=0.5):
    """
    计算蒸馏损失，包括硬标签交叉熵损失和软标签的KL散度损失。
    
    student_outputs: 学生模型的输出
    teacher_outputs: 教师模型的输出
    temperature: 温度参数，用于平滑softmax输出
    alpha: 权重系数，控制硬标签损失与软标签损失的比例
    """
    # 计算软标签损失（KL散度）
    teacher_probs = nn.functional.softmax(teacher_outputs / temperature, dim=1)
    student_probs = nn.functional.softmax(student_outputs / temperature, dim=1)
    
    distillation_loss = nn.functional.kl_div(
        nn.functional.log_softmax(student_outputs / temperature, dim=1),
        teacher_probs,
        reduction='batchmean'
    ) * (temperature ** 2)

    # 计算硬标签损失（交叉熵损失）
    hard_loss = nn.CrossEntropyLoss()(student_outputs, teacher_outputs.argmax(dim=1))

    # 总损失 = 蒸馏损失 + 硬标签损失
    return alpha * distillation_loss + (1 - alpha) * hard_loss

In [45]:
# ========== 模型定义 ========== 
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model('resnet50', pretrained=True)
        self.encoder.fc = nn.Identity()

    def forward(self, x):
        return self.encoder(x)

class NIR_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten())

    def forward(self, x):
        return self.conv_blocks(x.unsqueeze(1))

class ContrastiveFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj_img = nn.Linear(2048, 256)
        self.proj_nir = nn.Linear(128, 256)

    def forward(self, img_feat, nir_feat):
        img_proj = self.proj_img(img_feat)
        nir_proj = self.proj_nir(nir_feat)
        return img_proj + nir_proj

class MultiModalClassifier(nn.Module):
    def __init__(self, image_encoder, nir_encoder, fusion, num_classes):
        super().__init__()
        self.image_encoder = image_encoder
        self.nir_encoder = nir_encoder
        self.fusion = fusion
        self.classifier = nn.Sequential(
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, num_classes))

    def forward(self, img, nir):
        img_feat = self.image_encoder(img)
        nir_feat = self.nir_encoder(nir)
        fused_feat = self.fusion(img_feat, nir_feat)
        logits = self.classifier(fused_feat)
        return logits, fused_feat


In [46]:
# ========== 自定义图像数据集 ==========
class EnhancedDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.images = self._load_images()

    def _load_images(self):
        images = []
        for cls in self.classes:
            cls_path = os.path.join(self.root_dir, cls)
            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    img_path = os.path.join(cls_path, img_name)
                    images.append((img_path, self.class_to_idx[cls]))
        print(f"共加载 {len(images)} 张图片，共 {len(self.classes)} 个类别。")
        return images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# ========== NIR 数据读取 ==========
def load_nir_from_mat(mat_path):
    from scipy.io import loadmat
    data = loadmat(mat_path)
    return torch.tensor(data['nir'], dtype=torch.float32)  # 假设为 [120, 波长数]


In [47]:
# ========== 权重采样器与损失权重 ==========
def get_weighted_sampler(dataset):
    label_list = [label for _, label in dataset]
    class_counts = Counter(label_list)
    num_samples = len(label_list)
    class_weights = {cls: num_samples / count for cls, count in class_counts.items()}
    weights = [class_weights[label] for label in label_list]
    sampler = WeightedRandomSampler(weights, num_samples, replacement=True)
    return sampler, class_weights

def get_weighted_loss(class_weights):
    weights = torch.tensor([class_weights[i] for i in range(len(class_weights))], dtype=torch.float32).cuda()
    return nn.CrossEntropyLoss(weight=weights)

# ========== 模型初始化 ==========
image_encoder = StudentModel().cuda()  # 使用学生模型
nir_encoder = NIR_Encoder()
fusion = ContrastiveFusion()
model = MultiModalClassifier(image_encoder, nir_encoder, fusion, num_classes=3).cuda()

# 初始化教师模型
teacher_model = TeacherModel().cuda()

# ========== 数据加载 ==========
transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])
train_dataset = EnhancedDataset(root_dir=r"L:\常惠林\萎凋\自然萎凋\原始", transform=transform)
nir_data = load_nir_from_mat(r"L:\常惠林\萎凋\NIR.mat")  # [120, 波段数]

# ========== 加权采样器和损失 ==========
sampler, class_weights = get_weighted_sampler(train_dataset)
train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
ce_loss = get_weighted_loss(class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)



c:\Users\enine\anaconda3\envs\changhl\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\enine\anaconda3\envs\changhl\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\enine/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:02<00:00, 22.3MB/s]


共加载 156 张图片，共 3 个类别。


In [48]:
def train_distillation(model, teacher_model, dataloader, epochs):
    model.train()
    teacher_model.eval()  # 教师模型设为评估模式
    
    for epoch in range(epochs):
        total_loss = 0
        for imgs, nirs, labels in dataloader:
            imgs, nirs, labels = imgs.cuda(), nirs.cuda(), labels.cuda()
            
            optimizer.zero_grad()
            
            # 前向传播
            logits, student_img_feat, teacher_img_feat = model(imgs, nirs)
            
            # 计算蒸馏损失
            loss = distillation_loss(logits, teacher_img_feat, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

In [49]:
train_distillation(model, train_loader, epochs=10)

TypeError: train_distillation() missing 1 required positional argument: 'dataloader'